# Readmission Risk Summary

Reads `medicare_claims.inpatient_claims` (see
`data-generators/medicare_inpatient_claims_ingest.ipynb`) and computes
30-day readmission risk by primary diagnosis — a classic health-system
operations question: which diagnoses drive the most costly repeat
admissions.

For each beneficiary's claims (ordered by admission date), the gap
between one discharge and the next admission is computed via a window
function; a gap of 30 days or less counts as a readmission.

Re-runnable: `mode("overwrite")`, so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "medicare_claims"
SOURCE_TABLE = "inpatient_claims"
TARGET_TABLE = "readmission_risk_summary_v1"
READMISSION_WINDOW_DAYS = 30

## Load bronze table

In [ ]:
df = spark.table(f"{DATABASE_NAME}.{SOURCE_TABLE}")

## Transform: flag 30-day readmissions

Per beneficiary, `lag()` over admission-date-ordered claims gives the
prior discharge date; the gap to the current admission determines whether
it's a readmission.

In [ ]:
admission_window = Window.partitionBy("DESYNPUF_ID").orderBy("CLM_ADMSN_DT")

claims_with_gap_df = (
    df
    .withColumn("prior_discharge_dt", F.lag("NCH_BENE_DSCHRG_DT").over(admission_window))
    .withColumn(
        "days_since_prior_discharge",
        F.datediff(
            F.to_date(F.col("CLM_ADMSN_DT").cast("string"), "yyyyMMdd"),
            F.to_date(F.col("prior_discharge_dt").cast("string"), "yyyyMMdd"),
        ),
    )
    .withColumn(
        "is_readmission",
        F.when(
            F.col("days_since_prior_discharge").isNotNull()
            & (F.col("days_since_prior_discharge") <= READMISSION_WINDOW_DAYS),
            1,
        ).otherwise(0),
    )
)

## Aggregate by primary diagnosis

In [ ]:
summary_df = (
    claims_with_gap_df
    .groupBy(F.col("ICD9_DGNS_CD_1").alias("primary_diagnosis_code"))
    .agg(
        F.count("*").alias("total_admissions"),
        F.sum("is_readmission").alias("readmissions_within_30d"),
        F.round(F.avg("CLM_PMT_AMT"), 2).alias("avg_claim_payment"),
    )
    .withColumn(
        "readmission_rate_pct",
        F.round(F.col("readmissions_within_30d") * 100.0 / F.col("total_admissions"), 1),
    )
    .orderBy(F.desc("total_admissions"))
)

## Persist as a managed Spark table

In [ ]:
summary_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TARGET_TABLE}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_diagnoses FROM {DATABASE_NAME}.{TARGET_TABLE}").show()
spark.sql(f"SELECT * FROM {DATABASE_NAME}.{TARGET_TABLE} ORDER BY total_admissions DESC LIMIT 10").show()